# Title: train DS creating

Creating train dataset for final version of 'prefilter' model development.

### Vision

**How the data will be saved**
- common HDF5 dataset with all the usual keys, mirroring `merged_mc`
- no norming
- the h5 'parts' from Grisha's train dataset will be obligatory included  
(path: `/home2/ivkhar/Baikal/data/normed/baikal_2020_sig-noise_mid-eq_normed.h5`)

**Why so?**
- pytorch dataset for reading from this structure already exists and works well:
    - interleaving batches and balancing classes
    - sequental reading (fast and managable)
- common within our team

### Output path:
- `/home2/albert/Baikal/data/baikal_mc2020_prefilter_train.h5`

In [1]:
import os
import logging
import time
from pathlib import Path
from typing import List, Optional

import polars as pl
import h5py
import numpy as np

In [ ]:
DEFAULT_H5_PATH = Path("/net/62/home3/ivkhar/Baikal/data/h5s/baikal_mc_merged.h5")
SOURCE_CATALOG = Path("./h5_catalogs/catalogs_mc_signoise_normed/train.parquet")

MUATM_CATALOG = Path("./h5_catalogs/catalogs_mc_merged/muatm_2020.parquet")
NUATM_CATALOG = Path("./h5_catalogs/catalogs_mc_merged/nuatm_2020.parquet")
NUE2_CATALOG = Path("./h5_catalogs/catalogs_mc_merged/nue2_2020.parquet")

OUTPUT_FILE_PATH =  Path("/home2/albert/Baikal/data/baikal_mc2020_prefilter_train.h5")

## Selecting parts from `...merged_mc.h5` to store

### Load parts, that to be included obligatory

In [3]:
# Lazy column loading
norm_train_ids = pl.scan_parquet(SOURCE_CATALOG).select("event_id").collect()
df_temp = norm_train_ids.with_columns(
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.first().alias("particle_type"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(1).cast(pl.Int32).alias("part_num"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(2).cast(pl.Int32).alias("part_event_id"),
)

parts_to_store = {
    'muatm': [],
    'nuatm': [],
    'nue2': []
}
parts_counts_to_store = {
    'muatm': None,
    'nuatm': None,
    'nue2': None
}
for ptype in parts_to_store:
    part_ids_with_counts = df_temp.filter(pl.col('particle_type')==ptype)['part_num'].value_counts().sort('part_num')
    parts_to_store[ptype] = part_ids_with_counts['part_num'].to_list()
    parts_counts_to_store[ptype] = part_ids_with_counts['count']
    print(f"For {ptype}:")
    print(f"\tnumber of parts: {part_ids_with_counts['count'].shape[0]}")
    print(f"\tnumber of events: {part_ids_with_counts['count'].sum()}")
    print(f"\tmean number of events: {part_ids_with_counts['count'].mean()}")

For muatm:
	number of parts: 374
	number of events: 10890482
	mean number of events: 29118.935828877005
For nuatm:
	number of parts: 187
	number of events: 6289473
	mean number of events: 33633.545454545456
For nue2:
	number of parts: 88
	number of events: 1664273
	mean number of events: 18912.19318181818


### Get numbers of parts to add more

In [4]:
nuatm_already_number = parts_counts_to_store['nuatm'].sum()
muatm_already_number = parts_counts_to_store['muatm'].sum()
nue2_already_number = parts_counts_to_store['nue2'].sum()
assert nuatm_already_number*2 > muatm_already_number
assert nuatm_already_number > nue2_already_number

mean_nuatm_per_part = parts_counts_to_store['nuatm'].mean()
mean_muatm_per_part = parts_counts_to_store['muatm'].mean()
mean_nue2_per_part = parts_counts_to_store['nue2'].mean()

NUM_PARTS_TO_ADD = {
    'muatm': int(np.round((nuatm_already_number*2 - muatm_already_number)/mean_muatm_per_part)),
    'nuatm': int(np.round((nuatm_already_number-nuatm_already_number)/mean_nuatm_per_part)),
    'nue2': int(np.round((nuatm_already_number - nue2_already_number)/mean_nue2_per_part)),
}
NUM_PARTS_TO_ADD

{'muatm': 58, 'nuatm': 0, 'nue2': 245}

### Randomly select parts to add from main catalogs

In [5]:
norm_train_ids = pl.scan_parquet(SOURCE_CATALOG).select("event_id").collect()
df_temp = norm_train_ids.with_columns(
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.first().alias("particle_type"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(1).cast(pl.Int32).alias("part_num"),
    pl.col("event_id").cast(pl.Utf8).str.split("_").list.get(2).cast(pl.Int32).alias("part_event_id"),
)

In [10]:
mu_parts_already: list = parts_to_store['muatm']
new_mu_df = pl.scan_parquet(MUATM_CATALOG).select('h5_part_num').group_by('h5_part_num').len().filter(~pl.col('h5_part_num').is_in(mu_parts_already)).collect()
new_mu_parts: list = new_mu_df['h5_part_num'].sample(fraction=1, shuffle=True)[:NUM_PARTS_TO_ADD['muatm']].to_list()

print(f"Required {nuatm_already_number*2 - muatm_already_number:,} events.")
print(f"Will collect: {new_mu_df.filter(pl.col('h5_part_num').is_in(new_mu_parts))['len'].sum():,}")

Required 1,688,464 events.
Will collect: 1,700,676


In [7]:
nuatm_parts_already: list = parts_to_store['nuatm']
new_nuatm_df = pl.scan_parquet(NUATM_CATALOG).select('h5_part_num').group_by('h5_part_num').len().filter(~pl.col('h5_part_num').is_in(nuatm_parts_already)).collect()
new_nuatm_parts: list = new_nuatm_df['h5_part_num'].sample(fraction=1, shuffle=True)[:NUM_PARTS_TO_ADD['nuatm']].to_list()

print(f"Required {0:,} events.")
print(f"Will collect: {new_nuatm_df.filter(pl.col('h5_part_num').is_in(new_nuatm_parts))['len'].sum():,}")

Required 0 events.
Will collect: 0


In [8]:
nue2_parts_already: list = parts_to_store['nue2']
new_nue2_df = pl.scan_parquet(NUE2_CATALOG).select('h5_part_num').group_by('h5_part_num').len().filter(~pl.col('h5_part_num').is_in(nue2_parts_already)).collect()
new_nue2_parts: list = new_nue2_df['h5_part_num'].sample(fraction=1, shuffle=True)[:NUM_PARTS_TO_ADD['nue2']].to_list()

print(f"Required {nuatm_already_number - nue2_already_number:,} events.")
print(f"Will collect: {new_nue2_df.filter(pl.col('h5_part_num').is_in(new_nue2_parts))['len'].sum():,}")

Required 4,625,200 events.
Will collect: 4,699,970


In [ ]:
parts_to_store_final = {}
parts_to_store_final['muatm'] = parts_to_store['muatm'] + new_mu_parts
parts_to_store_final['nuatm'] = parts_to_store['nuatm'] + new_nuatm_parts
parts_to_store_final['nue2'] = parts_to_store['nue2'] + new_nue2_parts

## Creating the ouput h5 data

In [11]:
PARTICLE_MAP = {
    'muatm': 'muatm_2020',
    'nuatm': 'nuatm_2020',
    'nue2': 'nue2_2020',
}

# Keys to copy per part
# Per-hit arrays (indexed by hit position within part):
HIT_KEYS = ["raw/data", "raw/labels", "raw/channels"]
# Per-event arrays (indexed by event position within part):
EVENT_KEYS = ["raw/ev_starts", "raw/cluster_ids", "ev_ids"]

with h5py.File(DEFAULT_H5_PATH, "r") as src, \
     h5py.File(OUTPUT_FILE_PATH, "w") as dst:

    total_parts = sum(len(v) for v in parts_to_store_final.values())
    done = 0

    for ptype_short, part_nums in parts_to_store_final.items():
        ptype_full = PARTICLE_MAP[ptype_short]
        src_grp = src[ptype_full]
        n_parts = len(part_nums)
        
        print(f"\n{'='*60}")
        print(f"Copying {ptype_full}: {n_parts} parts")
        print(f"{'='*60}")
        
        t0 = time.time()
        total_events = 0
        total_hits = 0
        
        for i, part_num in enumerate(sorted(part_nums)):
            pk = f"part_{part_num}"
            
            for key in HIT_KEYS + EVENT_KEYS:
                # e.g. src["muatm_2020"]["raw"]["data"]["part_1000"]["data"]
                src_ds = src_grp
                for p in key.split("/"):
                    src_ds = src_ds[p]
                src_ds = src_ds[pk]["data"]
                
                dst_path = f"{ptype_full}/{key}/{pk}/data"
                dst.create_dataset(dst_path, data=src_ds[:])
            
            n_ev = src_grp["ev_ids"][pk]["data"].shape[0]
            n_hits = src_grp["raw"]["data"][pk]["data"].shape[0]
            total_events += n_ev
            total_hits += n_hits
            done += 1
            
            if (i + 1) % 20 == 0 or i == n_parts - 1:
                elapsed = time.time() - t0
                rate = (i + 1) / elapsed if elapsed > 0 else 1
                eta = (n_parts - i - 1) / rate
                print(f"  [{i+1}/{n_parts}] {pk}: {n_ev:,} events, {n_hits:,} hits "
                      f"({elapsed:.0f}s elapsed, ETA {eta:.0f}s) "
                      f"[{done}/{total_parts} total]")
        
        print(f"  Done: {total_events:,.0f} events, {total_hits:,.0f} hits")

fsize = os.path.getsize(OUTPUT_FILE_PATH)
print(f"\nOutput file: {OUTPUT_FILE_PATH}")
print(f"Size: {fsize / 1e9:.2f} GB")

# Write description file
desc_path = OUTPUT_FILE_PATH.with_suffix(".txt")
with open(desc_path, "w") as f:
    f.write(f"File: {OUTPUT_FILE_PATH.name}\n")
    f.write(f"Created: {time.strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write("Purpose:\n")
    f.write("  Training dataset for 'prefilter' neutrino classification model.\n")
    f.write("  Uncompressed for fast random-access reads during training.\n\n")
    f.write("Source:\n")
    f.write(f"  {DEFAULT_H5_PATH}\n")
    f.write("  Parts selected to include all events from Grisha's normed train split\n")
    f.write(f"  ({SOURCE_CATALOG}),\n")
    f.write("  plus additional parts to balance particle types.\n\n")
    f.write("Particle types and part counts:\n")
    for ps, pf in PARTICLE_MAP.items():
        n = len(parts_to_store_final[ps])
        f.write(f"  {pf}: {n} parts\n")
    f.write(f"\nHDF5 structure (per particle type, per part):\n")
    f.write(f"  ev_ids/part_N/data        — event IDs (bytes string)\n")
    f.write(f"  raw/ev_starts/part_N/data  — cumulative hit indices (n_events+1,)\n")
    f.write(f"  raw/data/part_N/data       — hit features (n_hits, 5): [amp, time, x, y, z]\n")
    f.write(f"  raw/labels/part_N/data     — per-hit signal/noise label (n_hits,)\n")
    f.write(f"  raw/channels/part_N/data   — per-hit channel ID (n_hits,)\n")
    f.write(f"  raw/cluster_ids/part_N/data — per-event cluster ID (n_events,)\n")
    f.write(f"\nCoordinates are cluster-centered (same as source).\n")
    f.write(f"No normalization applied.\n")
    f.write(f"No compression (uncompressed for fast I/O).\n")

print(f"Description saved: {desc_path}")


Copying muatm_2020: 432 parts
  [20/432] part_10381: 32,733 events, 2,046,460 hits (25s elapsed, ETA 508s) [20/952 total]
  [40/432] part_11317: 32,731 events, 2,048,274 hits (50s elapsed, ETA 491s) [40/952 total]
  [60/432] part_12052: 32,819 events, 2,054,311 hits (75s elapsed, ETA 467s) [60/952 total]
  [80/432] part_12961: 32,598 events, 2,034,405 hits (101s elapsed, ETA 445s) [80/952 total]
  [100/432] part_13807: 32,424 events, 2,025,367 hits (127s elapsed, ETA 420s) [100/952 total]
  [120/432] part_14351: 32,862 events, 2,053,589 hits (149s elapsed, ETA 388s) [120/952 total]
  [140/432] part_22639: 30,317 events, 2,367,703 hits (174s elapsed, ETA 362s) [140/952 total]
  [160/432] part_23411: 30,508 events, 2,379,452 hits (199s elapsed, ETA 339s) [160/952 total]
  [180/432] part_24362: 30,167 events, 2,355,936 hits (218s elapsed, ETA 305s) [180/952 total]
  [200/432] part_25557: 30,290 events, 2,361,495 hits (245s elapsed, ETA 284s) [200/952 total]
  [220/432] part_26560: 30,383

In [12]:
# Quick sanity check: compare a few parts between source and output
import random

with h5py.File(DEFAULT_H5_PATH, "r") as src, \
     h5py.File(OUTPUT_FILE_PATH, "r") as dst:

    print("Output file keys:", list(dst.keys()))
    print()

    for ptype_short, ptype_full in PARTICLE_MAP.items():
        part_nums = parts_to_store_final[ptype_short]
        # Pick 2 random parts to verify
        check_parts = random.sample(part_nums, min(2, len(part_nums)))

        n_parts_dst = len(dst[ptype_full]["ev_ids"].keys())
        print(f"{ptype_full}: {n_parts_dst} parts in output "
              f"(expected {len(part_nums)}), checking {check_parts}...")

        assert n_parts_dst == len(part_nums), \
            f"Part count mismatch: {n_parts_dst} vs {len(part_nums)}"

        for pn in check_parts:
            pk = f"part_{pn}"
            for key in ["raw/data", "raw/labels", "raw/channels",
                         "raw/ev_starts", "raw/cluster_ids", "ev_ids"]:
                src_path = f"{ptype_full}/{key}/{pk}/data"
                src_arr = src[src_path][:]
                dst_arr = dst[src_path][:]
                assert src_arr.shape == dst_arr.shape, \
                    f"{src_path}: shape {src_arr.shape} vs {dst_arr.shape}"
                assert np.array_equal(src_arr, dst_arr), \
                    f"{src_path}: data mismatch!"

            n_ev = dst[f"{ptype_full}/ev_ids/{pk}/data"].shape[0]
            n_hits = dst[f"{ptype_full}/raw/data/{pk}/data"].shape[0]
            print(f"  {pk}: {n_ev:,} events, {n_hits:,} hits — OK")

            # Verify no compression
            ds = dst[f"{ptype_full}/raw/data/{pk}/data"]
            assert ds.compression is None, f"Expected uncompressed, got {ds.compression}"

    print("\nAll checks passed!")

Output file keys: ['muatm_2020', 'nuatm_2020', 'nue2_2020']

muatm_2020: 432 parts in output (expected 432), checking [13999, 25509]...
  part_13999: 32,691 events, 2,043,799 hits — OK
  part_25509: 30,066 events, 2,347,330 hits — OK
nuatm_2020: 187 parts in output (expected 187), checking [1201, 4420]...
  part_1201: 39,527 events, 2,393,891 hits — OK
  part_4420: 29,790 events, 2,600,440 hits — OK
nue2_2020: 333 parts in output (expected 333), checking [2038, 4075]...
  part_2038: 19,686 events, 1,642,741 hits — OK
  part_4075: 16,632 events, 1,564,758 hits — OK

All checks passed!
